# 1.1 Build the Cohorts

**Class 1:** pull the data for our study from the All of Us dataset (CDR) with SQL.

**Study question:** how do people **with** and **without diabetes** differ in demographics, PTSD care, and cardiometabolic measurements?

**Study population** (both groups):

- Age 25–50, alive
- Race: White, Black, or Asian
- Ethnicity: Hispanic or not Hispanic
- Gender: Man or Woman
- Flagged **diabetes = Yes** (any diabetes mellitus diagnosis) or **No**

**We pull three tables:**

| Table | Variables | Saved as |
|---|---|---|
| Demographics | date of birth, gender, race, ethnicity, diabetes | `person.csv` |
| Survey | two PTSD questions: still seeing a doctor? currently treated? | `survey.csv` |
| Measurements | BMI, systolic blood pressure, LDL cholesterol, HbA1c | `measurement.csv` |

> 💡 **Run the queries once.** They cost money and take a few minutes. After that, later notebooks read the saved CSVs.

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data or counts of 1 to 20. Clear outputs before you share or commit this notebook.

In [ ]:
source("aou_setup.R")  # settings, packages, and helper functions

## The study population in SQL

The population is defined once, in `aou_setup.R`, as two **common table expressions** (CTEs): `diabetes_people` and `cohort`. Every query below starts with them. Read it through:

- `concept_ancestor` finds **all** diabetes codes under *diabetes mellitus* (concept 201820), so specific types like type 2 diabetes are included
- The concept IDs in `WHERE` are the **standard OMOP concepts** for each race, ethnicity, and gender

In [ ]:
cat(cohort_sql())

## Query 1: Demographics

We join the `person` table to `concept` to turn concept IDs into readable names.

In [ ]:
person_sql <- str_glue("{cohort_sql()}
SELECT
  c.person_id,
  c.diabetes,
  DATE(p.birth_datetime) AS date_of_birth,
  g.concept_name AS gender,
  r.concept_name AS race,
  e.concept_name AS ethnicity
FROM cohort c
JOIN `{CURRENT_DATASET}.person` p ON c.person_id = p.person_id
LEFT JOIN `{CURRENT_DATASET}.concept` g ON p.gender_concept_id    = g.concept_id
LEFT JOIN `{CURRENT_DATASET}.concept` r ON p.race_concept_id      = r.concept_id
LEFT JOIN `{CURRENT_DATASET}.concept` e ON p.ethnicity_concept_id = e.concept_id
")

person <- run_query(person_sql)
glimpse(person)

In [ ]:
save_data(person, "person.csv")

## Query 2: Survey answers

Survey answers are in the `observation` table:

- `observation_source_concept_id` is the **question**
- `value_source_concept_id` is the **answer**

The two PTSD questions are concepts 43528846 and 43530361.

In [ ]:
survey_sql <- str_glue("{cohort_sql()}
SELECT
  o.person_id,
  q.concept_name AS question,
  a.concept_name AS answer
FROM `{CURRENT_DATASET}.observation` o
JOIN cohort c ON o.person_id = c.person_id
LEFT JOIN `{CURRENT_DATASET}.concept` q ON o.observation_source_concept_id = q.concept_id
LEFT JOIN `{CURRENT_DATASET}.concept` a ON o.value_source_concept_id       = a.concept_id
WHERE o.observation_source_concept_id IN (43528846, 43530361)
")

survey <- run_query(survey_sql)
glimpse(survey)

Check the question and answer wording. Notebook 2.0 matches on it:

In [ ]:
survey %>% distinct(question, answer)

In [ ]:
save_data(survey, "survey.csv")

## Query 3: Measurements

The `measurement` table holds labs and vitals. Again we use `concept_ancestor` to include every more specific code under:

| Concept ID | Measurement |
|---|---|
| 3038553 | BMI |
| 3004249, 4152194 | Systolic blood pressure |
| 40795800 | LDL cholesterol |
| 3004410 | HbA1c |

This is the biggest query: **millions of rows** (one per measurement). Notebook 2.0 keeps the most recent value per person.

In [ ]:
measurement_sql <- str_glue("{cohort_sql()}
SELECT
  m.person_id,
  m.measurement_concept_id,
  mc.concept_name AS standard_concept_name,
  m.measurement_datetime,
  m.value_as_number
FROM `{CURRENT_DATASET}.measurement` m
JOIN cohort c ON m.person_id = c.person_id
LEFT JOIN `{CURRENT_DATASET}.concept` mc ON m.measurement_concept_id = mc.concept_id
WHERE m.value_as_number IS NOT NULL
  AND m.measurement_concept_id IN (
    SELECT descendant_concept_id
    FROM `{CURRENT_DATASET}.concept_ancestor`
    WHERE ancestor_concept_id IN (3038553, 3004249, 4152194, 40795800, 3004410)
  )
")

measurement <- run_query(measurement_sql)
glimpse(measurement)

In [ ]:
save_data(measurement, "measurement.csv")

## Check what you saved

These files are now in your bucket's `data/` folder. You won't need to run the queries again.

In [ ]:
list.files(DATA_DIR)

---

## Alternative: use a Data Explorer export instead

You can build the same two groups with **Data Explorer** instead of writing SQL:

1. Add **two cohorts**, using the criteria at the top of this notebook: one **with** diabetes (Condition: *Diabetes mellitus*) and one **without** (exclude it)
2. For each cohort, **Save data snapshot** with *Demographics*, the two PTSD survey questions, and the four measurements, as an **R notebook**
3. Run each exported notebook once. It saves CSV files to your bucket under `bq_exports/`
4. Then use the code below to read them and save them in the **same format** as above, so notebooks 2.0 to 3.4 work unchanged

The exported **person** table uses different column names (e.g. `T_DISP_gender`). This renames them:

In [ ]:
# Find the exported files (they're in folders named by table and date)
list.files(file.path(BUCKET_DIR, "bq_exports"), recursive = TRUE, pattern = "\\.csv$")

In [ ]:
# Read every CSV in one export folder into a single data frame
read_export <- function(folder) {
  list.files(folder, pattern = "\\.csv$", full.names = TRUE) %>%
    map_dfr(read_csv, show_col_types = FALSE)
}

# EDIT these paths: one person export folder per cohort
person_study   <- read_export(file.path(BUCKET_DIR, "bq_exports", "EDIT/study/person/folder"))
person_control <- read_export(file.path(BUCKET_DIR, "bq_exports", "EDIT/control/person/folder"))

person <- bind_rows(
  person_study   %>% mutate(diabetes = "Yes"),
  person_control %>% mutate(diabetes = "No")
) %>%
  transmute(person_id, diabetes,
            date_of_birth = as.Date(birth_datetime),
            gender    = T_DISP_gender,
            race      = T_DISP_race,
            ethnicity = T_DISP_ethnicity)

# save_data(person, "person.csv")   # uncomment to save

Do the same for the **survey** and **measurement** exports: read both cohorts, combine them, and rename columns to match the SQL versions above (`person_id`, `question`, `answer`; and `person_id`, `measurement_concept_id`, `standard_concept_name`, `measurement_datetime`, `value_as_number`). Use `names()` to see what the export calls them.